In [1]:
library(tidyr)
library(dplyr)
library(ggplot2)
library(patchwork)
library(data.table)

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/module_projection_fxns.R")

setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/frontal_cortex")

options(repr.plot.width=15, repr.plot.height=8, repr.plot.res=150)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


Loading required package: future


Attaching package: ‘reshape2’


The following objects are masked from ‘package:data.table’:

    dcast, melt


The following object is masked from ‘package:tidyr’:

    smiths




Here I visualize module genes to try to decide on how to clean up the bulk data prior to running FM again (to hopefully boost signal of certain cell types)

In [2]:
mod_def <- "TopModPosBC"

## Prep bulk data

In [ ]:
bulk_expr <- fread("GTEx_frontal_cortex_counts_TMMF_SampleNetworks/All_10-58-02/GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed.csv", data.table=FALSE)
colnames(bulk_expr)[1] <- "Gene"
rownames(bulk_expr) <- bulk_expr$Gene
bulk_expr <- bulk_expr[,-1]

## Prep single-cell data

In [ ]:
library(Matrix)

sc_data_source <- "ma_2022_counts_normalized"
sc_expr <- readMM("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/ma_2022/human/matrix.mtx")
barcodes <- readLines("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/ma_2022/human/barcodes.tsv")
features <- read.table("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/ma_2022/human/features.tsv", sep="\t")
cell_meta <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/ma_2022/human/cell_metadata.csv", data.table=FALSE)

colnames(sc_expr) <- barcodes
rownames(sc_expr) <- features[,2]

all.equal(colnames(sc_expr), cell_meta[,1])

In [ ]:
ctype_assignment_vec <- cell_meta$subclass

total_expr <- colSums(sc_expr)
sc_expr_norm <- sc_expr %*% Diagonal(x = 1 / total_expr) * 1e4
colnames(sc_expr_norm) <- colnames(sc_expr)

## Test

In [ ]:
# kME_path <- "/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/frontal_cortex/GTEx_frontal_cortex_counts_TMMF_rd2_All_200_outliers_removed_mergeParam0.93_subsetCutoff68.044_Modules/Bicor-None_signum0.795_minSize5_merge_ME_0.93_14114/kME_table_07-38-38.csv"
# mod <- "darkgreen"
# mod_genes <- get_mod_genes(kME_path, mod, mod_def)
# pdf("test.pdf")
# plot_gene_projections(sc_expr_norm, mod_genes, ctype_assignment_vec, plot_title="", plot_sub="", target_species=NULL)
# dev.off()

## Visualize module genes

### Select enrichments

In [ ]:
# Curated marker enrichements

enrich_source <- "GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed_mergeParam0.93_subsetCutoff68.086_Modules_Curated_EA_top_Qval_mods"

top_mods_df <- fread(paste0("data/EA/", enrich_source, ".csv"), data.table=FALSE)

max_qval <- .05

top_mods_df <- top_mods_df[top_mods_df$Qval < max_qval,]

dim(top_mods_df)

In [ ]:
# MO marker enrichments

enrich_source <- "GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed_mergeParam0.93_subsetCutoff68.086_Modules_MO_1354sets_EA_top_Qval_mods"

max_qval <- 1e-25

top_mods_df <- fread(paste0("data/EA/", enrich_source, ".csv"), data.table=FALSE)
top_mods_df <- top_mods_df[top_mods_df$Qval < max_qval,]

dim(top_mods_df)

In [ ]:
# Curated marker FGSEA enrichments

enrich_source <- "GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed_mergeParam0.93_subsetCutoff68.086_Modules_Curated_PC1_FGSEA_top_Qval_mods"

top_mods_df <- fread(paste0("data/FGSEA/", enrich_source, ".csv"), data.table=FALSE)
colnames(top_mods_df)[grep("Mod", colnames(top_mods_df))] <- "Module"
top_mods_df$Qval <- top_mods_df$padj

dim(top_mods_df)

[1] 24 16

### Plot

In [ ]:
outdir <- paste0("figures/module_projections/", sc_data_source, "/", mod_def)

if (!dir.exists(outdir)) {
    dir.create(outdir, recursive=TRUE)
}

filename <- paste0(outdir, "/", enrich_source, ".pdf")

max_genes <- 15

pdf(file=filename, width=10, height=8)

for (i in 1:nrow(top_mods_df)) {
    print(paste("Module", i))

    mod <- top_mods_df$Module[i]
    kME_path <- top_mods_df$kME_path[i]
    mod_genes <- get_mod_genes(kME_path, mod, mod_def)

    plot_title <- paste(
        top_mods_df$Cell_type[i], mod_def, 
        "\n", mod, top_mods_df$Network[i]
    ) 
    plot_sub <- paste("Qval:", round(top_mods_df$Qval[i], 4))

    # Plot gene expression over bulk samples
    
    mod_genes_subset <- na.omit(mod_genes[1:max_genes])
    plot_gene_expr_over_samples(bulk_expr, mod_genes_subset, plot_title, plot_sub, target_species=NULL)

    # Plot module genes in single cell data

    plot_gene_projections(sc_expr_norm, mod_genes, ctype_assignment_vec, plot_title, plot_sub, target_species=NULL)
}

dev.off()